In [62]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

In [63]:
URLS = [
    "https://www.google.com",      # Funciona
    "https://this-url-no-existe.com", # Falla (DNS error)
    "https://httpbin.org/delay/5"  # Lento (Timeout)
]

### Manejo de errores
Si pasara un error en el procesamiento de alguna pagina el script quedaria colgado esperando eternamente. 
as_completed permite que no se detengan los demas que si haya completado su mision.

In [64]:
def check_health(url):
    resp = requests.get(url, timeout=3)
    return f"{url}: Status {resp.status_code}"

In [65]:
def err_handling_example():
    with ThreadPoolExecutor(max_workers=3) as executor:
        future_map = {executor.submit(check_health, url): url for url in URLS}

        for future in as_completed(future_map):
            url = future_map[future]
            try:
                data = future.result()
                print(f"Datos: {data}")

            except Exception as e:
                print(f"Hubo un fallo en: {url}: {type(e).__name__} - {e}") 
                    # {type(e).__name__} obtiene el nombre de la excepción (TimeoutError, ValueError, etc.

    with ThreadPoolExecutor(max_workers=3) as executor:
        future_map = {executor.submit(check_health, url): url for url in URLS}
        print("\n--- Manejo de errores sin as_completed")
        for future in future_map:
            url = future_map[future]
            try:
                data = future.result()
                print(f"Datos: {data}")

            except Exception as e:
                print(f"Hubo un fallo en: {url}: {type(e).__name__} - {e}") 


In [66]:
err_handling_example()

Hubo un fallo en: https://this-url-no-existe.com: ConnectionError - HTTPSConnectionPool(host='this-url-no-existe.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='this-url-no-existe.com', port=443): Failed to resolve 'this-url-no-existe.com' ([Errno -2] Name or service not known)"))
Datos: https://www.google.com: Status 200
Hubo un fallo en: https://httpbin.org/delay/5: ReadTimeout - HTTPSConnectionPool(host='httpbin.org', port=443): Read timed out. (read timeout=3)

--- Manejo de errores sin as_completed
Datos: https://www.google.com: Status 200
Hubo un fallo en: https://this-url-no-existe.com: ConnectionError - HTTPSConnectionPool(host='this-url-no-existe.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='this-url-no-existe.com', port=443): Failed to resolve 'this-url-no-existe.com' ([Errno -2] Name or service not known)"))
Hubo un fallo en: https://httpbin.org/delay/5: ReadTime